# Hyperparameter Tuning

In this notebook, we will find suitable values of the regularization parameter alpha for Ridge and Lasso Regression.

The alpha parameter controls the strength of regularization.

In [1]:
from sklearn.linear_model import RidgeCV, LassoCV

In [2]:
alpha_=[0.001, 0.01, 0.1, 1, 10, 100]
ridge_cv = RidgeCV(
    alphas=alpha_,
    cv=5
)

In [13]:
import pandas as pd
import numpy as np
df=pd.read_csv("../data/insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
X=df.drop("charges",axis=1)
y=df["charges"]

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
y_train.shape

(1070,)

In [6]:
X_train["sex"]=X_train["sex"].map({"female":1,"male":0})
X_train["smoker"]=X_train["smoker"].map({"yes":1,"no":0})
X_test["sex"]=X_test["sex"].map({"female":1,"male":0})
X_test["smoker"]=X_test["smoker"].map({"yes":1,"no":0})
X_train

,age,sex,bmi,children,smoker,region
560,46,1,19.950,2,0,northwest
1285,47,1,24.320,0,0,northeast
1142,52,1,24.860,0,0,southeast
969,39,1,34.320,5,0,southeast
486,54,1,21.470,3,0,northwest
...,...,...,...,...,...,...
1095,18,1,31.350,4,0,northeast
1130,39,1,23.870,5,0,southeast
1294,58,0,25.175,0,0,northeast
860,37,1,47.600,2,1,southwest


In [7]:
X_train=pd.get_dummies(X_train,columns=["region"],drop_first=True,dtype=int)
X_test=pd.get_dummies(X_test,columns=["region"],drop_first=True,dtype=int)
X_train

,age,sex,bmi,children,smoker,region_northwest,region_southeast,region_southwest
560,46,1,19.950,2,0,1,0,0
1285,47,1,24.320,0,0,0,0,0
1142,52,1,24.860,0,0,0,1,0
969,39,1,34.320,5,0,0,1,0
486,54,1,21.470,3,0,1,0,0
...,...,...,...,...,...,...,...,...
1095,18,1,31.350,4,0,0,0,0
1130,39,1,23.870,5,0,0,1,0
1294,58,0,25.175,0,0,0,0,0
860,37,1,47.600,2,1,0,0,1


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train[['age','bmi','children']]=scaler.fit_transform(
    X_train[['age','bmi','children']]
)
X_test[['age','bmi','children']]=scaler.transform(
    X_test[['age','bmi','children']]
)
X_train

,age,sex,bmi,children,smoker,region_northwest,region_southeast,region_southwest
560,0.472227,1,-1.756525,0.734336,0,1,0,0
1285,0.543313,1,-1.033082,-0.911192,0,0,0,0
1142,0.898745,1,-0.943687,-0.911192,0,0,1,0
969,-0.025379,1,0.622393,3.202629,0,0,1,0
486,1.040918,1,-1.504893,1.557100,0,1,0,0
...,...,...,...,...,...,...,...,...
1095,-1.518194,1,0.130717,2.379865,0,0,0,0
1130,-0.025379,1,-1.107579,3.202629,0,0,1,0
1294,1.325264,0,-0.891539,-0.911192,0,0,0,0
860,-0.167551,1,2.820864,0.734336,1,0,0,1


In [9]:
ridge_cv.fit(X_train, y_train)

,alphas,"[0.001, 0.01, ...]"
,fit_intercept,True
,scoring,None
,cv,5
,gcv_mode,None
,store_cv_results,False
,alpha_per_target,False


In [10]:
print("Best alpha:", ridge_cv.alpha_)

Best alpha: 1.0


In [11]:
ridge_cv_pred = ridge_cv.predict(X_test)

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ridge_cv_mae = mean_absolute_error(y_test, ridge_cv_pred)
ridge_cv_mse = mean_squared_error(y_test, ridge_cv_pred)
ridge_cv_rmse = np.sqrt(ridge_cv_mse)
ridge_cv_r2 = r2_score(y_test, ridge_cv_pred)
print("MAE:", ridge_cv_mae)
print("MSE:", ridge_cv_mse)
print("RMSE:", ridge_cv_rmse)
print("Test R²:", ridge_cv_r2)

MAE: 4193.19535293527
MSE: 33645393.49385556
RMSE: 5800.464937731764
Test R²: 0.7832807188145148


## Ridge Hyperparameter Tuning Conclusion

RidgeCV tested multiple alpha values using 5-fold cross-validation and selected alpha = 1.0 as the best value based on cross-validation performance.

When evaluated on the unseen test set, RidgeCV produced a Test R² of approximately 0.7833, while the manually selected alpha = 0.1 produced approximately 0.7836.

The difference is very small, showing that both regularization strengths perform similarly on this dataset. Hyperparameter tuning does not guarantee an improvement on a particular test set because the test set is not used during hyperparameter selection.

In [16]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(
    ridge_cv,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)
print("CV scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())

CV scores: [0.71571638 0.80193951 0.72318198 0.65868783 0.76623502]
Mean CV R²: 0.7331521431579194
